In [ ]:
import pandas as pd
import os
import glob
import re
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_dir = '/content/drive/MyDrive/WhisperData'

if not os.path.isdir(base_dir):
    raise FileNotFoundError(f"Could not find {base_dir}")

In [ ]:
assess_path = os.path.join(base_dir, 'assess.csv')
asses = pd.read_csv(assess_path)
rec_paths = sorted(glob.glob(os.path.join(base_dir, 'rec[1-6].csv')))
recc_list = [pd.read_csv(p) for p in rec_paths]
recc = pd.concat(recc_list, ignore_index=True)
costRegEx = re.compile(r".*cost$|^[EW]\d+$")
cost_cols = [c for c in asses.columns if costRegEx.match(c)]
print(cost_cols)
asses["cost"] = asses[cost_cols].sum(axis=1)
cols_idx = list(asses.columns)
start_idx = cols_idx.index("EC_plant_cost")
end_idx   = cols_idx.index("W6_plant_usage")
costUse_cols_to_drop = cols_idx[start_idx : end_idx + 1]
print(f"Dropping {len(costUse_cols_to_drop)} columns:")
for c in costUse_cols_to_drop:
    print("  •", c)
asses= asses.drop(columns=costUse_cols_to_drop)

print("\nRemaining columns:")
print(asses.columns.tolist())
asses.head(5)

['EC_plant_cost', 'ED_plant_cost', 'EF_plant_cost', 'E2_plant_cost', 'E3_plant_cost', 'E4_plant_cost', 'E5_plant_cost', 'E6_plant_cost', 'E7_plant_cost', 'E8_plant_cost', 'E9_plant_cost', 'E10_plant_cost', 'E11_plant_cost', 'E12_plant_cost', 'W0_plant_cost', 'W1_plant_cost', 'W2_plant_cost', 'W3_plant_cost', 'W4_plant_cost', 'W5_plant_cost', 'W6_plant_cost']
Dropping 41 columns:
  • EC_plant_cost
  • EC_plant_usage
  • ED_plant_cost
  • ED_plant_usage
  • EF_plant_cost
  • E2_plant_cost
  • E2_plant_usage
  • E3_plant_cost
  • E3_plant_usage
  • E4_plant_cost
  • E4_plant_usage
  • E5_plant_cost
  • E5_plant_usage
  • E6_plant_cost
  • E6_plant_usage
  • E7_plant_cost
  • E7_plant_usage
  • E8_plant_cost
  • E8_plant_usage
  • E9_plant_cost
  • E9_plant_usage
  • E10_plant_cost
  • E10_plant_usage
  • E11_plant_cost
  • E11_plant_usage
  • E12_plant_cost
  • E12_plant_usage
  • W0_plant_cost
  • W0_plant_usage
  • W1_plant_cost
  • W1_plant_usage
  • W2_plant_cost
  • W2_plant_usage
  

,ID,CENTER,FY,SIC,NAICS,STATE,SALES,EMPLOYEES,PLANT_AREA,PRODUCTS,PRODUNITS,PRODLEVEL,PRODHOURS,NUMARS,cost
0,AM0001,AM,1987,3671.0,NaN,TX,33900000.0,206.0,NaN,MICRO & MINI COMPUTERS,1.0,310.0,2250.0,7,184985.0
1,AM0002,AM,1987,2761.0,NaN,TX,25000000.0,156.0,NaN,BUSINESS FORMS,NaN,NaN,2250.0,9,338359.0
2,AM0003,AM,1987,3494.0,NaN,TX,15000000.0,200.0,NaN,SAFETY JOINTS & VALVES,NaN,NaN,2250.0,8,139480.0
3,AM0004,AM,1987,3713.0,NaN,TX,4200000.0,75.0,NaN,TRUCKS BEDS & TRAILERS,1.0,1000.0,2340.0,11,68986.0
4,AM0005,AM,1987,2024.0,NaN,TX,12000000.0,39.0,NaN,ICE CREAM,5.0,2000.0,2080.0,7,185065.0


**asses is the asses dataframe**

In [ ]:
saved_cols = [c for c in recc.columns if c.upper().endswith("SAVED")]
print(saved_cols)
imp_idx = recc.columns.get_loc("IMPCOST")
recc["Savings"] = recc[saved_cols].sum(axis=1)
keep_cols = list(recc.columns[:imp_idx+1]) + ["Savings"]
recc= recc[keep_cols]


['PSAVED', 'SSAVED', 'TSAVED', 'QSAVED']


In [ ]:
recc.head(10)

,SUPERID,ID,AR_NUMBER,APPCODE,ARC2,IMPSTATUS,IMPCOST,Savings
0,AM000101,AM0001,1,NaN,2.8114,N,15000.0,1828.0
1,AM000102,AM0001,2,NaN,2.7142,N,189.0,663.0
2,AM000103,AM0001,3,NaN,2.7111,N,398.0,950.0
3,AM000104,AM0001,4,NaN,2.7447,I,354.0,544.0
4,AM000105,AM0001,5,NaN,2.7233,N,15.0,270.0
5,AM000106,AM0001,6,NaN,2.7111,N,NaN,183.0
6,AM000107,AM0001,7,NaN,2.7111,N,NaN,41.0
7,AM000201,AM0002,1,NaN,2.6221,I,5500.0,11479.0
8,AM000202,AM0002,2,NaN,2.7226,I,60000.0,95084.0
9,AM000203,AM0002,3,NaN,2.6212,I,NaN,205.0


**recc is the dataframe before collapsing into a single col per assessment**

In [ ]:
cols_to_drop = ["SUPERID", "AR_NUMBER", "APPCODE"]
recc_collapsed = recc.drop(columns=cols_to_drop)

In [ ]:
recc_collapsed.head(10)

,ID,ARC2,IMPSTATUS,IMPCOST,Savings
0,AM0001,2.8114,N,15000.0,1828.0
1,AM0001,2.7142,N,189.0,663.0
2,AM0001,2.7111,N,398.0,950.0
3,AM0001,2.7447,I,354.0,544.0
4,AM0001,2.7233,N,15.0,270.0
5,AM0001,2.7111,N,NaN,183.0
6,AM0001,2.7111,N,NaN,41.0
7,AM0002,2.6221,I,5500.0,11479.0
8,AM0002,2.7226,I,60000.0,95084.0
9,AM0002,2.6212,I,NaN,205.0


In [ ]:
agg_dict = {
    "ARC2"       : ("ARC2",       lambda x: ",".join(x.dropna().astype(str))),
    "IMPSTATUS"  : ("IMPSTATUS",  lambda x: ",".join(x.dropna().astype(str))),
    "TOTAL_IMPCOST"    : ("IMPCOST",    "sum"),
    "TOTAL_Savings"    : ("Savings",    "sum"),
    "num_reccs"  : ("ID",         "size"),
}
recc_collapsed = (
    recc_collapsed
    .groupby("ID", as_index=False)
    .agg(**agg_dict)
)
recc_collapsed["ROI_YRS"] = recc_collapsed["TOTAL_IMPCOST"] / recc_collapsed["TOTAL_Savings"]

recc_collapsed = recc_collapsed[[
    "ID", "num_reccs", "ARC2", "IMPSTATUS",
    "TOTAL_IMPCOST", "TOTAL_Savings", "ROI_YRS"
]]

print(recc_collapsed.head())




       ID  num_reccs                                               ARC2  \
0  AM0001          7   2.8114,2.7142,2.7111,2.7447,2.7233,2.7111,2.7111   
1  AM0002          9  2.6221,2.7226,2.6212,2.4111,2.7142,2.7142,2.71...   
2  AM0003          8  2.3212,2.7224,2.7111,2.4111,2.7142,2.7142,2.24...   
3  AM0004         11  2.6218,2.7142,2.7142,2.4236,2.7142,2.6232,2.62...   
4  AM0005          7   2.7142,2.7142,2.7142,2.2449,2.4111,2.1233,2.8112   

               IMPSTATUS  TOTAL_IMPCOST  TOTAL_Savings   ROI_YRS  
0          N,N,N,I,N,N,N        15956.0         4479.0  3.562402  
1      I,I,I,N,I,I,I,I,I        73676.0       109532.0  0.672644  
2        N,N,N,N,N,I,N,I        15724.0        16001.0  0.982689  
3  I,I,I,I,N,N,N,I,N,N,N        22429.0        17607.0  1.273868  
4          I,I,N,N,N,I,N         7453.0         3402.0  2.190770  


**Merging the two dataframes and creating a custom dataset for further analysis**

In [ ]:
merged = (
    asses
    .merge(
        recc_collapsed[[
            "ID",
            "num_reccs",
            "ARC2",
            "IMPSTATUS",
            "TOTAL_IMPCOST",
            "TOTAL_Savings",
            "ROI_YRS"
        ]],
        on="ID",
        how="left"
    )
)
cols = [
    "ID", "CENTER", "FY", "SIC", "NAICS", "STATE",
    "SALES", "EMPLOYEES", "PLANT_AREA", "cost",
    "TOTAL_IMPCOST", "TOTAL_Savings", "ROI_YRS"
]
rest = [c for c in merged.columns if c not in cols]

merged = merged[cols + rest]
rename_map = {
    "SALES":      "ANNUAL_SALES",
    "cost":    "ANNUAL_ELECTRICITY_COST",
    "TOTAL_Savings":    "TOTAL_RECC_SAVINGS",
    "TOTAL_IMPCOST":    "TOTAL_RECC_IMP_COST"

}

merged = merged.rename(columns=rename_map)



In [ ]:
merged

,ID,CENTER,FY,SIC,NAICS,STATE,ANNUAL_SALES,EMPLOYEES,PLANT_AREA,ANNUAL_ELECTRICITY_COST,...,TOTAL_RECC_SAVINGS,ROI_YRS,PRODUCTS,PRODUNITS,PRODLEVEL,PRODHOURS,NUMARS,num_reccs,ARC2,IMPSTATUS
0,AM0001,AM,1987,3671.0,NaN,TX,33900000.0,206.0,NaN,184985.0,...,4479.0,3.562402,MICRO & MINI COMPUTERS,1.0,310.0,2250.0,7,7,"2.8114,2.7142,2.7111,2.7447,2.7233,2.7111,2.7111","N,N,N,I,N,N,N"
1,AM0002,AM,1987,2761.0,NaN,TX,25000000.0,156.0,NaN,338359.0,...,109532.0,0.672644,BUSINESS FORMS,NaN,NaN,2250.0,9,9,"2.6221,2.7226,2.6212,2.4111,2.7142,2.7142,2.71...","I,I,I,N,I,I,I,I,I"
2,AM0003,AM,1987,3494.0,NaN,TX,15000000.0,200.0,NaN,139480.0,...,16001.0,0.982689,SAFETY JOINTS & VALVES,NaN,NaN,2250.0,8,8,"2.3212,2.7224,2.7111,2.4111,2.7142,2.7142,2.24...","N,N,N,N,N,I,N,I"
3,AM0004,AM,1987,3713.0,NaN,TX,4200000.0,75.0,NaN,68986.0,...,17607.0,1.273868,TRUCKS BEDS & TRAILERS,1.0,1000.0,2340.0,11,11,"2.6218,2.7142,2.7142,2.4236,2.7142,2.6232,2.62...","I,I,I,I,N,N,N,I,N,N,N"
4,AM0005,AM,1987,2024.0,NaN,TX,12000000.0,39.0,NaN,185065.0,...,3402.0,2.190770,ICE CREAM,5.0,2000.0,2080.0,7,7,"2.7142,2.7142,2.7142,2.2449,2.4111,2.1233,2.8112","I,I,N,N,N,I,N"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22129,WV0679,WV,2025,3231.0,327215.0,WV,8000000.0,50.0,118467.0,209396.0,...,4451.0,1.135251,Glass Display Screens,3.0,22000.0,2080.0,4,4,"2.4236,2.7142,2.4231,2.4239",
22130,WV0680,WV,2025,3532.0,333922.0,WV,73000000.0,321.0,96411.0,386595.0,...,123484.0,1.460489,Conveyor Accessories,2.0,650000.0,8736.0,11,11,"2.4146,2.2511,2.7314,2.4322,2.4224,2.7135,2.71...",
22131,WV0681,WV,2025,3462.0,332111.0,PA,100000000.0,175.0,280000.0,1833148.0,...,59002.0,1.086743,Iron and Steel Forgings,2.0,100000000.0,8736.0,8,8,"2.4146,2.4151,2.1233,2.7142,2.4236,2.2511,2.41...",
22132,WV0682,WV,2025,2834.0,325212.0,VA,99000000.0,80.0,310000.0,1840821.0,...,281280.0,0.754775,"Feminine wash & cloths, Glycerin suppositories",2.0,5500000.0,7200.0,15,15,"2.4146,3.4151,2.7142,2.4157,2.2511,2.7135,2.26...",


In [ ]:
merged.to_csv("itac_compact.csv", index=False)
